# OMI Real Estate Quotations — Data Quality & Exploratory Analysis

This notebook builds a reproducible analytical dataset from the semiannual **OMI (Osservatorio del Mercato Immobiliare)** quotations published by the *Agenzia delle Entrate*.

### Analytical objective

Move from raw OMI releases to a validated dataset that can support: temporal market analysis, geographic comparisons, quotation-range analysis, property-category analysis, and future joins with transaction volumes.

### Pipeline

**Raw CSV files → consolidation → data quality → cleaning → validation → feature engineering → EDA → insights**

**Source:** Agenzia delle Entrate — Osservatorio del Mercato Immobiliare.


## 1. Setup

The notebook keeps the analytical stack lightweight: `pandas` for data preparation, `numpy` for numerical transformations and `matplotlib` for visualization. The project root is resolved from the notebook location so the code does not depend on a specific working directory.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'quotations'
assert RAW_DIR.exists(), f'Raw quotations directory not found: {RAW_DIR}'

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 2. Load and consolidate the raw OMI releases

OMI quotations are supplied as one semicolon-separated CSV per semester. Each file is loaded independently so that unexpected files can be identified early and the reference period is retained explicitly.


In [ ]:
files = sorted(RAW_DIR.glob('omi_quotations_*.csv'))
if not files:
    raise FileNotFoundError(f'No OMI quotation files found in {RAW_DIR}')

records = []
invalid_files = []

for path in files:
    parts = path.stem.rsplit('_', 2)
    if len(parts) != 3 or parts[1] not in {'S1', 'S2'}:
        invalid_files.append(path.name)
        continue

    year = int(parts[1])
    semester = parts[2]
    df_part = pd.read_csv(path, sep=';', low_memory=False)
    df_part['reference_year'] = year
    df_part['reference_semester'] = semester
    df_part['reference_period'] = f'{year}-{semester}'
    records.append(df_part)

if invalid_files:
    print('Files skipped because their name does not match the expected convention:')
    print(invalid_files)

omi = pd.concat(records, ignore_index=True)
print(f'Files loaded: {len(records):,}')
print(f'Rows consolidated: {len(omi):,}')
print(f'Columns: {omi.shape[1]:,}')


## 3. Initial dataset inspection

Inspect the schema, temporal coverage and raw structure before changing any values.


In [ ]:
display(omi.head())
display(omi.dtypes.to_frame('dtype'))
print('Raw shape:', omi.shape)
print('Duplicate rows:', omi.duplicated().sum())

display(
    omi['reference_period']
    .value_counts()
    .sort_index()
    .rename_axis('reference_period')
    .reset_index(name='rows')
)


## 4. Data quality assessment

Missing values are measured before deciding how to handle them. A missing quotation is not automatically equivalent to a zero quotation.


In [ ]:
missing = (
    omi.isna().mean().mul(100).sort_values(ascending=False).rename('missing_pct').to_frame()
)
display(missing.head(25))


In [ ]:
technical_columns = [column for column in omi.columns if column.startswith('Unnamed:')]
technical_check = {
    column: {'all_missing': omi[column].isna().all(), 'non_null': int(omi[column].notna().sum())}
    for column in technical_columns
}
display(pd.DataFrame(technical_check).T)

for column in technical_columns:
    if omi[column].isna().all():
        omi = omi.drop(columns=column)


### Numeric quotation fields

Quotation columns are converted explicitly. Malformed values become visible as missing values instead of causing silent downstream errors.


In [ ]:
numeric_columns = ['Compr_min', 'Compr_max', 'Loc_min', 'Loc_max']
for column in numeric_columns:
    if column in omi.columns:
        omi[column] = pd.to_numeric(omi[column], errors='coerce')

display(omi[numeric_columns].describe().T)


## 5. Structural validation

The following checks test assumptions that should hold for an OMI quotation record: minimum values must not exceed maximum values, quotations must not be negative, and the semester must be valid.


In [ ]:
validation = pd.DataFrame({
    'check': [
        'Compr_min > Compr_max', 'Loc_min > Loc_max',
        'Negative purchase quotation', 'Negative rental quotation', 'Invalid semester'
    ],
    'violations': [
        int((omi['Compr_min'] > omi['Compr_max']).sum()),
        int((omi['Loc_min'] > omi['Loc_max']).sum()),
        int((omi['Compr_min'] < 0).sum() + (omi['Compr_max'] < 0).sum()),
        int((omi['Loc_min'] < 0).sum() + (omi['Loc_max'] < 0).sum()),
        int((~omi['reference_semester'].isin(['S1', 'S2'])).sum()),
    ],
})
display(validation)


In [ ]:
candidate_key = ['reference_period', 'Comune_ISTAT', 'Fascia', 'Zona', 'Descr_Tipologia', 'Stato']
available_key = [column for column in candidate_key if column in omi.columns]
duplicate_key_rows = omi.duplicated(subset=available_key, keep=False).sum() if available_key else 0
print('Candidate analytical key:', available_key)
print('Rows participating in duplicate candidate keys:', duplicate_key_rows)


## 6. Cleaning and standardisation

Cleaning is conservative: text fields are trimmed, empty strings become missing values, and only empty technical columns are removed. Valid zero values are preserved. No quotation values are imputed.


In [ ]:
text_columns = omi.select_dtypes(include='object').columns
for column in text_columns:
    omi[column] = omi[column].str.strip()
omi[text_columns] = omi[text_columns].replace({'': pd.NA})

omi['reference_date'] = pd.to_datetime({
    'year': omi['reference_year'],
    'month': np.where(omi['reference_semester'].eq('S1'), 6, 12),
    'day': np.where(omi['reference_semester'].eq('S1'), 30, 31),
})

omi = omi.sort_values(['reference_date', 'Regione', 'Prov', 'Comune_descrizione'], kind='stable').reset_index(drop=True)
print('Cleaned shape:', omi.shape)


## 7. Feature engineering

The raw OMI data provide minimum and maximum quotations. We derive midpoint and spread measures for descriptive analysis. The midpoint is **not** an official OMI average price.


In [ ]:
omi['Compr_mid'] = omi[['Compr_min', 'Compr_max']].mean(axis=1)
omi['Loc_mid'] = omi[['Loc_min', 'Loc_max']].mean(axis=1)
omi['Compr_spread'] = omi['Compr_max'] - omi['Compr_min']
omi['Loc_spread'] = omi['Loc_max'] - omi['Loc_min']
omi['Compr_spread_pct'] = omi['Compr_spread'].div(omi['Compr_mid'].replace(0, np.nan)).mul(100)
omi['Loc_spread_pct'] = omi['Loc_spread'].div(omi['Loc_mid'].replace(0, np.nan)).mul(100)
display(omi[['Compr_min','Compr_max','Compr_mid','Compr_spread','Compr_spread_pct','Loc_min','Loc_max','Loc_mid','Loc_spread','Loc_spread_pct']].describe().T)


## 8. Temporal coverage

A continuous sequence of semesters is checked explicitly. This helps detect incomplete downloads before trend analysis.


In [ ]:
periods = omi[['reference_year','reference_semester','reference_period']].drop_duplicates().sort_values(['reference_year','reference_semester'])
observed_periods = set(periods['reference_period'])
years = range(int(periods['reference_year'].min()), int(periods['reference_year'].max()) + 1)
expected_periods = {f'{year}-S{semester}' for year in years for semester in (1, 2)}
missing_periods = sorted(expected_periods - observed_periods)
print(f'Observed periods: {len(observed_periods):,}')
print(f'Expected periods in range: {len(expected_periods):,}')
print('Missing periods:', missing_periods if missing_periods else 'None')


## 9. Geographic coverage

The OMI hierarchy allows analysis at several levels: **Area territoriale → Regione → Provincia → Comune → Zona OMI**.


In [ ]:
geographic_summary = pd.DataFrame({
    'regions': [omi['Regione'].nunique()],
    'provinces': [omi['Prov'].nunique()],
    'municipalities': [omi['Comune_ISTAT'].nunique()],
    'omi_zones': [omi['Zona'].nunique()],
})
display(geographic_summary)

regional_coverage = (
    omi.groupby('Regione', dropna=False)
    .agg(municipalities=('Comune_ISTAT','nunique'), zones=('Zona','nunique'), observations=('reference_period','size'))
    .sort_values('municipalities', ascending=False)
)
display(regional_coverage.head(20))


## 10. Property categories and conditions

OMI quotations are segmented by property type and condition. Observation counts help distinguish genuine market differences from differences in coverage.


In [ ]:
for column in ['Descr_Tipologia', 'Stato']:
    summary = omi[column].value_counts(dropna=False).rename_axis(column).reset_index(name='observations')
    display(summary.head(20))


## 11. Purchase quotation analysis

The first price analysis focuses on residential property descriptions. The filter is intentionally explicit and can later be replaced by a formal OMI typology mapping.


In [ ]:
residential_mask = omi['Descr_Tipologia'].astype('string').str.contains('abitazion|villa', case=False, na=False)
residential = omi.loc[residential_mask].copy()
print(f'Residential observations: {len(residential):,}')

national_trend = (
    residential.groupby('reference_date', as_index=False)
    .agg(median_compr_mid=('Compr_mid','median'), mean_compr_mid=('Compr_mid','mean'), observations=('Compr_mid','count'))
)
display(national_trend.tail(10))


### Purchase quotation trend

The median is used as the primary descriptive statistic because OMI observations span municipalities and micro-zones with very different price levels.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(national_trend['reference_date'], national_trend['median_compr_mid'], marker='o', linewidth=2)
ax.set_title('OMI Residential Purchase Quotations — Median Midpoint')
ax.set_xlabel('Reference period')
ax.set_ylabel('€ / m²')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 12. Geographic price comparison

A portfolio analysis should not stop at a national average. Regional market levels are compared at the latest available period, with the aggregation logic shown explicitly.


In [ ]:
latest_date = residential['reference_date'].max()
latest_regional = (
    residential.loc[residential['reference_date'].eq(latest_date)]
    .groupby('Regione', as_index=False)
    .agg(median_price_m2=('Compr_mid','median'), observations=('Compr_mid','count'), municipalities=('Comune_ISTAT','nunique'))
    .sort_values('median_price_m2', ascending=False)
)
display(latest_regional.head(15))


In [ ]:
top_regions = latest_regional.head(10).sort_values('median_price_m2')
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_regions['Regione'], top_regions['median_price_m2'])
ax.set_title('Top Regions by Median OMI Residential Purchase Quotation')
ax.set_xlabel('€ / m²')
ax.set_ylabel('')
fig.tight_layout()
plt.show()


## 13. Quotation-range dispersion

The OMI quotation is an interval rather than a point estimate. Relative spread provides information about the width of the reported market range.


In [ ]:
spread_summary = (
    residential.groupby('reference_date', as_index=False)
    .agg(median_spread_pct=('Compr_spread_pct','median'), p75_spread_pct=('Compr_spread_pct', lambda s: s.quantile(0.75)))
)
display(spread_summary.tail(10))


## 14. Final data-quality summary

The final checks provide a compact, auditable hand-off point for the next notebook.


In [ ]:
final_quality = pd.DataFrame({
    'metric': ['rows','columns','periods','duplicate rows','purchase min > max','rental min > max','negative purchase quotations','negative rental quotations'],
    'value': [
        len(omi), omi.shape[1], omi['reference_period'].nunique(), int(omi.duplicated().sum()),
        int((omi['Compr_min'] > omi['Compr_max']).sum()), int((omi['Loc_min'] > omi['Loc_max']).sum()),
        int((omi[['Compr_min','Compr_max']] < 0).sum().sum()), int((omi[['Loc_min','Loc_max']] < 0).sum().sum())
    ],
})
display(final_quality)


## 15. Key takeaways and limitations

### What this notebook establishes

- Raw semiannual OMI files are consolidated into one reproducible analytical dataset.
- Data-quality assumptions are tested explicitly before analysis.
- Purchase and rental quotation ranges are preserved, while midpoint and spread metrics are clearly labelled as derived measures.
- Temporal and geographic coverage are quantified.
- Residential purchase quotations can be compared consistently across periods and regions.

### Important limitations

- OMI quotations are intervals, not transaction-level prices.
- The midpoint is a derived descriptive measure, not an official OMI average.
- Observations are heterogeneous across municipalities, zones, property types and conditions.
- Missing rental quotations must not automatically be interpreted as zero.
- Cross-dataset analysis with transaction volumes requires a carefully defined geographic and temporal key.

### Next notebook

**Notebook 02 — OMI Transactions** will analyse transaction volumes and prepare the variables required to combine market prices with market activity.
